In [7]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI

In [2]:

load_dotenv() # .env 파일을 환경변수로 등록

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', "neo4j") # 기본값 neo4j

In [ ]:
# Neo4jGraph : Neo4j 연결정보를 받아서, 그래프 조회와 스키마 확인 기능을 하는 래퍼 클래스
graph = Neo4jGraph(
    url = NEO4J_URI,
    username = NEO4J_USERNAME,
    password = NEO4J_PASSWORD,
    database = NEO4J_DATABASE
)

print("LangChain과 Neo4j 연결 성공!")

LangChain과 Neo4j 연결 성공!


In [ ]:
graph.refresh_schema() # 스키마 새로고침 (그래프 구조 변경시 실행)

print(graph.schema) # 노드 레이블, 속성, 관계 유형과 방향

Node properties:
Student {age: INTEGER, name: STRING, student_id: INTEGER}
Course {name: STRING, course_id: INTEGER, level: STRING, duration: INTEGER}
Instructor {name: STRING, career: INTEGER, instructor_id: INTEGER}
Category {name: STRING, category_id: INTEGER}
Relationship properties:
ENROLLED_IN {score: INTEGER, enrolled_at: DATE}
The relationships:
(:Student)-[:ENROLLED_IN]->(:Course)
(:Course)-[:BELONGS_TO]->(:Category)
(:Instructor)-[:TEACHES]->(:Course)


In [5]:
query = """
MATCH (student:Student)-[:ENROLLED_IN]->(course:Course)
RETURN
    student.name AS student_name,
    course.name AS course_name,
    student.student_id AS student_id,
    course.course_id AS course_id
ORDER BY student_id, course_id
"""
result = graph.query(query)

result

[{'student_name': '홍길동',
  'course_name': 'Python',
  'student_id': 1,
  'course_id': 101},
 {'student_name': '홍길동',
  'course_name': 'Data Analysis',
  'student_id': 1,
  'course_id': 104},
 {'student_name': '김영희',
  'course_name': 'Database',
  'student_id': 2,
  'course_id': 102},
 {'student_name': '김영희',
  'course_name': 'Machine Learning',
  'student_id': 2,
  'course_id': 103},
 {'student_name': '이민수',
  'course_name': 'Python',
  'student_id': 3,
  'course_id': 101},
 {'student_name': '이민수',
  'course_name': 'Data Analysis',
  'student_id': 3,
  'course_id': 104},
 {'student_name': '박서연',
  'course_name': 'Machine Learning',
  'student_id': 4,
  'course_id': 103},
 {'student_name': '박서연',
  'course_name': 'Deep Learning',
  'student_id': 4,
  'course_id': 105},
 {'student_name': '최준호',
  'course_name': 'Database',
  'student_id': 5,
  'course_id': 102},
 {'student_name': '최준호',
  'course_name': 'LangChain',
  'student_id': 5,
  'course_id': 106}]

In [8]:
llm = ChatOpenAI(
    model = os.getenv('OPENAI_MODEL'),
    temperature=0 # DB의 정보만 가져올 것이므로 창의성은 0 (결정론적인 답변)
)

In [9]:
# LLM과 Neo4j 연결하여 자연어 질문을 Cypher로 변환하고 답변하는 체인
chain = GraphCypherQAChain.from_llm(
    llm = llm,      # Cypher 생성 및 최종 답변 llm
    graph = graph,  # 참조할 Neo4j 그래프 객체
    verbose = True, # 로그 출력
    validate_cypher= True, # 생성된 Cypher 검증
    return_intermediate_steps= True, # 중간과정 함께 반환
    top_k= 10,      # 조회결과 10개
    allow_dangerous_requests = True  # DB 쿼리 실행 위험성 확인
)

In [10]:
question = 'Python 강의를 수강하는 학생을 알려줘'

response = chain.invoke({"query": question})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
WHERE c.name = 'Python'
RETURN s.student_id, s.name;

Full Context:
[{'s.student_id': 3, 's.name': '이민수'}, {'s.student_id': 1, 's.name': '홍길동'}]

> Finished chain.


{'query': 'Python 강의를 수강하는 학생을 알려줘',
 'result': 'Python 강의를 수강하는 학생은 이민수와 홍길동입니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)\nWHERE c.name = 'Python'\nRETURN s.student_id, s.name;\n"},
  {'context': [{'s.student_id': 3, 's.name': '이민수'},
    {'s.student_id': 1, 's.name': '홍길동'}]}]}

In [ ]:
response['result'] # 최종 답변

'Python 강의를 수강하는 학생은 이민수와 홍길동입니다.'

In [12]:
response['intermediate_steps'] # 중간 과정 확인

[{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)\nWHERE c.name = 'Python'\nRETURN s.student_id, s.name;\n"},
 {'context': [{'s.student_id': 3, 's.name': '이민수'},
   {'s.student_id': 1, 's.name': '홍길동'}]}]

In [ ]:
def ask_graph(question: str) -> dict:
    if not question.strip():
        return ValueError('질문을 입력하셔야 합니다!')

    response = chain.invoke({"query": question})

    print(f"[질문] {question}")
    print(f"[최종 답변] {response['result']}")

    # 중간과정 추가시
    for step in response.get('intermediate_steps',[]):
        if "query" in step:
            print(f"[생성된 Cypher] {step['query']}")
        if "context" in step:
            print(f"[조회 결과] {step['context']}")

    return response

In [17]:
ask_graph("인공지능 카테고리에 속한 강의를 알려줘")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Course)-[:BELONGS_TO]->(cat:Category {name: '인공지능'})
RETURN c;

Full Context:
[{'c': {'duration': 48, 'course_id': 103, 'level': '입문', 'name': 'Machine Learning'}}, {'c': {'duration': 52, 'course_id': 105, 'level': '중급', 'name': 'Deep Learning'}}, {'c': {'duration': 32, 'course_id': 106, 'level': '중급', 'name': 'LangChain'}}]

> Finished chain.
[질문] 인공지능 카테고리에 속한 강의를 알려줘
[최종 답변] 인공지능 카테고리에 속한 강의는 Machine Learning, Deep Learning, LangChain입니다.
[생성된 Cypher] MATCH (c:Course)-[:BELONGS_TO]->(cat:Category {name: '인공지능'})
RETURN c;

[조회 결과] [{'c': {'duration': 48, 'course_id': 103, 'level': '입문', 'name': 'Machine Learning'}}, {'c': {'duration': 52, 'course_id': 105, 'level': '중급', 'name': 'Deep Learning'}}, {'c': {'duration': 32, 'course_id': 106, 'level': '중급', 'name': 'LangChain'}}]


{'query': '인공지능 카테고리에 속한 강의를 알려줘',
 'result': '인공지능 카테고리에 속한 강의는 Machine Learning, Deep Learning, LangChain입니다.',
 'intermediate_steps': [{'query': "MATCH (c:Course)-[:BELONGS_TO]->(cat:Category {name: '인공지능'})\nRETURN c;\n"},
  {'context': [{'c': {'duration': 48,
      'course_id': 103,
      'level': '입문',
      'name': 'Machine Learning'}},
    {'c': {'duration': 52,
      'course_id': 105,
      'level': '중급',
      'name': 'Deep Learning'}},
    {'c': {'duration': 32,
      'course_id': 106,
      'level': '중급',
      'name': 'LangChain'}}]}]}

In [18]:
ask_graph("수강생이 가장 많은 강의 알려줘")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
RETURN c.name AS course_name, count(s) AS student_count
ORDER BY student_count DESC
LIMIT 1;
Full Context:
[{'course_name': 'Machine Learning', 'student_count': 2}]

> Finished chain.
[질문] 수강생이 가장 많은 강의 알려줘
[최종 답변] 수강생이 가장 많은 강의는 **Machine Learning**이며, 수강생은 **2명**입니다.
[생성된 Cypher] MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
RETURN c.name AS course_name, count(s) AS student_count
ORDER BY student_count DESC
LIMIT 1;
[조회 결과] [{'course_name': 'Machine Learning', 'student_count': 2}]


{'query': '수강생이 가장 많은 강의 알려줘',
 'result': '수강생이 가장 많은 강의는 **Machine Learning**이며, 수강생은 **2명**입니다.',
 'intermediate_steps': [{'query': 'MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)\nRETURN c.name AS course_name, count(s) AS student_count\nORDER BY student_count DESC\nLIMIT 1;'},
  {'context': [{'course_name': 'Machine Learning', 'student_count': 2}]}]}

In [20]:
ask_graph('Capybara의 강의를 수강하는 학생들을 알려줘')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (i:Instructor {name: 'Capybara'})-[:TEACHES]->(c:Course)<-[:ENROLLED_IN]-(s:Student)
RETURN s.name, s.student_id;
Full Context:
[{'s.name': '이민수', 's.student_id': 3}, {'s.name': '홍길동', 's.student_id': 1}, {'s.name': '김영희', 's.student_id': 2}, {'s.name': '최준호', 's.student_id': 5}]

> Finished chain.
[질문] Capybara의 강의를 수강하는 학생들을 알려줘
[최종 답변] 이민수, 홍길동, 김영희, 최준호 학생입니다.
[생성된 Cypher] MATCH (i:Instructor {name: 'Capybara'})-[:TEACHES]->(c:Course)<-[:ENROLLED_IN]-(s:Student)
RETURN s.name, s.student_id;
[조회 결과] [{'s.name': '이민수', 's.student_id': 3}, {'s.name': '홍길동', 's.student_id': 1}, {'s.name': '김영희', 's.student_id': 2}, {'s.name': '최준호', 's.student_id': 5}]


{'query': 'Capybara의 강의를 수강하는 학생들을 알려줘',
 'result': '이민수, 홍길동, 김영희, 최준호 학생입니다.',
 'intermediate_steps': [{'query': "MATCH (i:Instructor {name: 'Capybara'})-[:TEACHES]->(c:Course)<-[:ENROLLED_IN]-(s:Student)\nRETURN s.name, s.student_id;"},
  {'context': [{'s.name': '이민수', 's.student_id': 3},
    {'s.name': '홍길동', 's.student_id': 1},
    {'s.name': '김영희', 's.student_id': 2},
    {'s.name': '최준호', 's.student_id': 5}]}]}